# File 02/2

# DESCRIPTION:
- Maps the places in column "place" to a dictionary containing place and region.
- Creates a new column "region"
## INPUT FILE:
"./OUTPUTS/dataframe_02_1.csv"
## OUTPUT FILE:
"./'OUTPUTS/dataframe_02_2.csv'"
## OPTIONAL OUTPUT_FILE 
"./OUTPUTS/VERBS_by_REGIONxCENTURY.xlsx" -> Table; is not needed afterwards

In [1]:
# third-party imports
import pandas as pd 

In [2]:
# read data frame from csv file
df = pd.read_csv("OUTPUTS/dataframe_02_1.csv", 
                 dtype={
                     "Russian Translation": "string",
                     "English Translation": "string"}
                )
df.head(2)

/var/folders/ps/86qydyw53xj72p_g1g25hfx00000gn/T/ipykernel_24369/960982334.py:2: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("OUTPUTS/dataframe_02_1.csv",


,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,POS,Morphology,Head ID,Relation,Presentation After,Russian Translation,English Translation,Type,century,exact,lang,source,place
0,mst,Mstislav’s letter,orv,189407,2157773,Се,се,I-,---------n,NaN,voc,,"вот, это","behold, here is",OR,12,1130,OR,NaN,Novgorod
1,mst,Mstislav’s letter,orv,189407,2157774,азъ,азъ,Pp,1s---mn--i,2157784.0,sub,,я,I,OR,12,1130,OR,NaN,Novgorod


In [3]:
# check for existing places in df 
list_place = list(df.place.unique())
df.place.unique()

array(['Novgorod', 'Pskov', 'Unknown', 'Staraja_Russa', 'Moscow',
       'Byzantine_Empire', 'Nizhny_Novgorod', 'Smolensk', 'Kiev',
       'Latvia', 'NorthernRussia', 'Tver'], dtype=object)

In [4]:
df.columns

Index(['File', 'Text Title', 'Language', 'Sentence ID', 'Token ID', 'Form',
       'Lemma', 'POS', 'Morphology', 'Head ID', 'Relation',
       'Presentation After', 'Russian Translation', 'English Translation',
       'Type', 'century', 'exact', 'lang', 'source', 'place'],
      dtype='object')

ERROR CHECKING: all columns in df["place"] should have valid values

In [5]:
# filter for place == NaN 
missing_place = df.loc[df['place'].isna(), ['File', 'Text Title']]
missing_place = missing_place.drop_duplicates()

assert len(missing_place) == 0, "ERROR: df['place'] still contains NaN values"
# execute if missing values exist 
# print(missing_place.to_string(index=False))

Create Mapping for "place" (in DataFrame) 

In [6]:
# - map the cities with their corresponding regions 
# - the regions are base on languages

region_map = {
    # PLACE           # REGION
    'Tver':           'East Slavic',
    'Moscow':         'East Slavic',
    'Novgorod':       'East Slavic',
    'Pskov':          'East Slavic',
    'Staraja_Russa':  'East Slavic',
    'NorthernRussia': 'East Slavic',
    'Kiev':           'East Slavic',
    'Nizhny_Novgorod':'East Slavic',
    'Smolensk':       'East Slavic',
    'Byzantine_Empire':'Byzantine_Empire',
    'Latvia':         'Latvia',
    None:             'Unknown', # if df["place"] is None
    'Unknown':        'Unknown' # if place is unknown, so is region 
}

In [7]:
# Create a list of unique places from df["place"]
list_place = list(df.place.unique())

# Keys in region_map, which are missing in list_place
missing_in_places = [region for region in region_map.keys() 
                     if region not in list_place]

if missing_in_places != [None]:
    print("region_map keys missing in list_place:", missing_in_places)
    raise AssertionError("ERROR: Missing places exist")

missing_in_map = [place for place in list_place 
                  if place not in region_map.keys()]

if missing_in_map: # if list is not empty 
    print("list_place entries missing in region_map:", missing_in_map)
    raise AssertionError(f"ERROR: Missing places:\n\n{missing_in_map}")

insert column "region" to DataFrame 

In [8]:
# Map region vals via "place" col (NaN is kept)
region_series = df["place"].map(region_map)

# Set unmapped Vals (including real NaNs) to "Unknown"
region_series = region_series.fillna("Unknown")

# Get idx of col "place" and insert mapped value to col "region_map"
place_idx = df.columns.get_loc("place")
df.insert(place_idx+1, "region", region_series)


validation for  df["region"]:

In [9]:
# assert column exists
assert "place" in df.columns, "ERROR: column 'places' does not exist"

# assert column has no NA / None
assert df["place"].notna().all(), "ERROR: column 'places' contains NA/None"

In [10]:
# Validation for  df["region"]:
assert not df["region"].isna().any(), "ERROR: region contains NaN values"

In [11]:
df[:2]

,File,Text Title,Language,Sentence ID,Token ID,Form,Lemma,POS,Morphology,Head ID,...,Presentation After,Russian Translation,English Translation,Type,century,exact,lang,source,place,region
0,mst,Mstislav’s letter,orv,189407,2157773,Се,се,I-,---------n,NaN,...,,"вот, это","behold, here is",OR,12,1130,OR,NaN,Novgorod,East Slavic
1,mst,Mstislav’s letter,orv,189407,2157774,азъ,азъ,Pp,1s---mn--i,2157784.0,...,,я,I,OR,12,1130,OR,NaN,Novgorod,East Slavic


In [12]:
df.to_csv('OUTPUTS/dataframe_02_2.csv', index=False)